# Customer Segmentation Project
### Thiranex Data Analytics Task 2

Customer segmentation using data cleaning, exploratory analysis, feature scaling and K-Means clustering.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans

In [ ]:
df = pd.read_csv('customer_data_raw.csv')
print(df.head())
print(df.shape)

In [ ]:
df = df.drop_duplicates().copy()
for col in ['AnnualIncome_k','SpendingScore']:
    df[col] = df[col].fillna(df[col].median())
df.describe()

In [ ]:
features = ['Age','AnnualIncome_k','SpendingScore','OrdersPerYear']
scaler = StandardScaler()
X = scaler.fit_transform(df[features])

In [ ]:
inertias=[]
for k in range(2,8):
    km=KMeans(n_clusters=k, random_state=42, n_init=10)
    km.fit(X)
    inertias.append(km.inertia_)
plt.plot(range(2,8), inertias, marker='o')
plt.xlabel('Number of Clusters')
plt.ylabel('Inertia')
plt.title('Elbow Method')
plt.show()

In [ ]:
kmeans = KMeans(n_clusters=4, random_state=42, n_init=10)
df['Cluster'] = kmeans.fit_predict(X)
df.groupby('Cluster')[features].mean().round(2)

In [ ]:
summary=df.groupby('Cluster')[features].mean()
names={}
for c,row in summary.iterrows():
    if row['AnnualIncome_k'] >= summary['AnnualIncome_k'].median() and row['SpendingScore'] >= summary['SpendingScore'].median():
        names[c]='High-Value Customers'
    elif row['AnnualIncome_k'] >= summary['AnnualIncome_k'].median():
        names[c]='High-Income Low-Spend'
    elif row['SpendingScore'] >= summary['SpendingScore'].median():
        names[c]='Regular Engaged Customers'
    else:
        names[c]='Low-Value Customers'
df['Segment']=df['Cluster'].map(names)
df.to_csv('customer_segments.csv', index=False)
df['Segment'].value_counts()

## Business Insights
- High-Value Customers are suitable for loyalty programs and personalized offers.
- High-Income Low-Spend customers are potential upsell/cross-sell targets.
- Regular Engaged Customers can be retained with relevant promotions.
- Low-Value Customers can be targeted with entry-level offers and engagement campaigns.